In [1]:
import requests
import matplotlib.pyplot as plt
import seaborn as sns  
import pandas as pd
import numpy as np
import time
from bs4 import BeautifulSoup
from mpl_toolkits.mplot3d import Axes3D

sns.set(style="whitegrid")

In [ ]:
years = [2019, 2020, 2021, 2022, 2023, 2024, 2026]
all_data = []

for year in years:
    print(f"Scraping {year}...")
    
    url = f"https://www.basketball-reference.com/leagues/NBA_{year}.html"
    tables = pd.read_html(url)
    
    east = tables[0]
    west = tables[1]
    
    standings = pd.concat([east, west])
    
    # Hapus header ulang
    standings = standings[standings.iloc[:,0] != standings.columns[0]]
    
    # Ambil kolom berdasarkan posisi index (lebih aman)
    temp = standings.iloc[:, [0,1,2,5,6]].copy()
    temp.columns = ["Team", "Wins", "Losses", "Off_PTS", "Def_PTS"]
    
    # Pastikan string
    temp["Team"] = temp["Team"].astype(str)
    
    # Buat label playoff
    temp["Playoff"] = temp["Team"].apply(lambda x: 1 if "*" in x else 0)
    temp["Team"] = temp["Team"].str.replace("*", "", regex=False)
    
    # Tambahkan season
    temp["Season"] = year
    
    all_data.append(temp)

team_df = pd.concat(all_data, ignore_index=True)

print("Total data:", team_df.shape)
team_df.head(50)

Scraping 2019...
Scraping 2020...
Scraping 2021...
Scraping 2022...
Scraping 2023...
Scraping 2024...
Scraping 2026...
Total data: (210, 7)


,Team,Wins,Losses,Off_PTS,Def_PTS,Playoff,Season
0,Milwaukee Bucks,60,22,118.1,109.3,1,2019
1,Toronto Raptors,58,24,114.4,108.4,1,2019
2,Philadelphia 76ers,51,31,115.2,112.5,1,2019
3,Boston Celtics,49,33,112.4,108.0,1,2019
4,Indiana Pacers,48,34,108.0,104.7,1,2019
5,Brooklyn Nets,42,40,112.2,112.3,1,2019
6,Orlando Magic,42,40,107.3,106.6,1,2019
7,Detroit Pistons,41,41,107.0,107.3,1,2019
8,Charlotte Hornets,39,43,110.7,111.8,0,2019
9,Miami Heat,39,43,105.7,105.9,0,2019


: 

In [ ]:
team_df["Off_PTS"] = pd.to_numeric(team_df["Off_PTS"])
team_df["Def_PTS"] = pd.to_numeric(team_df["Def_PTS"])

team_df["Def_Index"] = -team_df["Def_PTS"]
team_df["Off_Index"] = team_df["Off_PTS"]

plt.figure(figsize=(8,6))

sns.scatterplot(
    data=team_df,
    x="Off_Index",
    y="Def_Index",
    hue="Playoff",
    style="Season",
    s=120
)

plt.title("Offense vs Defense and Playoff Qualification (2021–2024)")
plt.xlabel("Offensive Points per Game")
plt.ylabel("Defensive Strength (Lower PA = Better)")

plt.show()